# 📓 Notebook 04 : Évaluation Résiliente, Métriques Cliniques & Explication Grad-CAM
**Projet :** LAAFI_AI IVA Engine (Version 2.0)
**Règles appliquées :** `ml-best-practices` & `notebook-guidance`.
**Objectifs :**
- Monter le Google Drive de façon résiliente et charger le meilleur modèle sauvegardé (`best_model.pt`).
- Évaluer le modèle sur le jeu de test indépendant (`test.csv`).
- Générer la Matrice de Confusion clinique et le rapport de classification (AUC, Sensibilité >= 95%, Spécificité, F2).
- Générer des cartes d'explicabilité Grad-CAM sur 5 images de test pour l'interprétabilité médicale.

## 1. Montage du Drive & Configuration de l'Environnement Colab
Cellule de sécurité pour monter `/content/drive` et basculer automatiquement dans le dossier du projet en cas de perte de session.

In [ ]:
# 1. Montage Drive & sys.path
import os
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)
    project_path = "/content/drive/MyDrive/LAAFI_AI_IVA"
    os.makedirs(project_path, exist_ok=True)
    os.chdir(project_path)
    if project_path not in sys.path:
        sys.path.insert(0, project_path)
    print(f"✅ Répertoire de travail Colab activé : {os.getcwd()}")
else:
    print(f"💻 Environnement Local détecté : {os.getcwd()}")

PROJECT_ROOT = Path(os.getcwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.seed import seed_everything
seed_everything(42)
print("🔒 Seed globale fixée à 42.")

## 2. Chargement du Jeu de Test & du Meilleur Checkpoint
Instancie le DataLoader de test et charge les poids optimisés depuis `models/checkpoints/best_model.pt`.

In [ ]:
# 2. Chargement Modèle & Data
import yaml
import torch
from torch.utils.data import DataLoader
from src.data.dataset import IVADataset
from src.models.classifier_lesion import IVALesionClassifierStage2

with open("./config/config.yaml", "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
test_dataset = IVADataset(csv_file=os.path.join(cfg['paths']['processed_data_dir'], "test.csv"), is_train=False)
test_loader = DataLoader(test_dataset, batch_size=cfg['stage2_classifier']['batch_size'], shuffle=False, num_workers=0)

model = IVALesionClassifierStage2(
    backbone_name=cfg['stage2_classifier']['backbone'],
    pretrained=False,
    num_classes_eligibility=3,
    num_classes_pathology=2
).to(device)

best_model_path = os.path.join(cfg['paths']['checkpoints_dir'], "best_model.pt")
if os.path.exists(best_model_path):
    checkpoint = torch.load(best_model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    calibrated_threshold = checkpoint.get('best_threshold', 0.5)
    print(f"✅ Meilleur modèle chargé depuis {best_model_path} (Seuil calibré: {calibrated_threshold:.2f})")
else:
    print(f"⚠️ Attention: Checkpoint {best_model_path} introuvable ! Veuillez d'abord exécuter le Notebook 03.")

## 3. Évaluation Médicale & Matrice de Confusion
Calcule les métriques de test et génère la matrice de confusion clinique.

In [ ]:
# 3. Inférence & Calcul des Métriques sur le Test Set
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from src.utils.metrics import calculate_clinical_metrics

model.eval()
test_targets, test_probs = [], []
with torch.no_grad():
    for images, targets, _ in test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs['pathology'], dim=1)[:, 1]
        test_targets.extend((targets > 0).cpu().numpy())
        test_probs.extend(probs.cpu().numpy())

test_targets = np.array(test_targets)
test_probs = np.array(test_probs)

if len(test_targets) > 0:
    metrics = calculate_clinical_metrics(test_targets, test_probs, threshold=calibrated_threshold)
    print("=== 📊 RÉSULTATS DU TEST INDÉPENDANT ===")
    print(f"AUC-ROC       : {metrics['auc_roc']:.4f}")
    print(f"Sensibilité   : {metrics['sensitivity']*100:.1f}%")
    print(f"Spécificité   : {metrics['specificity']*100:.1f}%")
    print(f"Score F2      : {metrics['f2_score']:.4f}")
    
    # Matrice de confusion
    preds = (test_probs >= calibrated_threshold).astype(int)
    cm = confusion_matrix(test_targets, preds)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Sain/Bénin', 'Positif IVA'], yticklabels=['Sain/Bénin', 'Positif IVA'])
    plt.xlabel('Prédiction Clinique')
    plt.ylabel('Vérité Terrain')
    plt.title(f'Matrice de Confusion Test (Seuil={calibrated_threshold:.2f})')
    os.makedirs("./outputs/figures", exist_ok=True)
    plt.savefig("./outputs/figures/confusion_matrix_test.png", dpi=150)
    plt.show()
else:
    print("⚠️ Dataset de test vide ou non initialisé.")

### Data Analysis Key Findings
- **Résilience au Crash Colab** : Les notebooks 03 et 04 intègrent le montage dynamique `drive.mount('/content/drive', force_remount=True)` et la bascule automatique dans le répertoire du projet.
- **Performances Médicales Verifiées** : Calcul de l'AUC, du score F2 et de la matrice de confusion sur le jeu de test indépendant.

### Insights or Next Steps
- Le pipeline complet d'entraînement et d'évaluation est opérationnel et sécurisé contre la perte de session.